# 📝 Transforming StatsBomb Events Into Player Stats

**Competition:** Premier League 2015/16   
**Dataset:** StatsBomb Open Data (events and lineups via `statsbombpy`)  
**Purpose:** Build one row per player who appeared in the season: minutes and starts from lineups, on-ball counts from events.  
**Methods:** Season match loop, lineup position windows for minutes, vectorised event flags, aggregation into a single table.  
**Author:** [Oscar Yu](https://www.linkedin.com/in/oscar-yu-cheuk-chun/)  

---



# 1. 📦 Imports & Setup

Constants for the open-data Premier League season, the output path, and the columns on the season table. `STAT_FIELDS` is the full catalogue. `appearances`, `starts`, and `minutes` come from lineups. Volume and mix counts come from events. Per 90, success rates, and mix shares are derived after the season totals and are not summed from matches.


In [1]:
# Data
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

# StatsBomb open data. Suppress the no-credentials warning.
import warnings
from statsbombpy import sb

try:
    from statsbombpy.api_client import NoAuthWarning

    warnings.filterwarnings("ignore", category=NoAuthWarning)
except ImportError:
    pass

# Premier League 2015/16 in the StatsBomb open-data catalogue.
COMPETITION_ID = 2
SEASON_ID = 27
SEASON_LABEL = "2015/2016"

DATA_PATH = Path("data/statsbomb/pl_1516_player_season_stats.csv")

# Season table catalogue (identity columns are added at write time).
# Lineups: appearances, starts, minutes. Events: volume and mix counts.
# Per 90, rates, and shares are derived after the season totals.
STAT_FIELDS = [
    # Lineups
    "appearances",
    "starts",
    "minutes",
    # Shot
    "goals",
    "xg",
    "shots",
    "shots_on_target",
    "shots_first_time",
    "shots_one_on_one",
    "shots_left_foot",
    "shots_right_foot",
    "shots_head",
    # Shot p90
    "goals_p90",
    "xg_p90",
    "shots_p90",
    "shots_on_target_p90",
    "shots_first_time_p90",
    "shots_one_on_one_p90",
    # Shot rates and shares
    "shot_accuracy",
    "conversion",
    "xg_per_shot",
    "goals_minus_xg",
    "shot_left_share",
    "shot_right_share",
    "shot_head_share",
    # Chance creation
    "assists",
    "key_passes",
    # Chance creation p90
    "assists_p90",
    "key_passes_p90",
    # Passing
    "passes",
    "passes_completed",
    "passes_left_foot",
    "passes_right_foot",
    "passes_head",
    "passes_ground",
    "passes_low",
    "passes_high",
    "crosses",
    "cut_backs",
    "switches",
    "through_balls",
    # Passing p90
    "passes_p90",
    "passes_completed_p90",
    "crosses_p90",
    "cut_backs_p90",
    "switches_p90",
    "through_balls_p90",
    # Passing rates and shares
    "pass_completion",
    "pass_left_share",
    "pass_right_share",
    "pass_head_share",
    "pass_ground_share",
    "pass_low_share",
    "pass_high_share",
    "cross_share",
    "cut_back_share",
    "switch_share",
    "through_ball_share",
    # Carries and dribbles
    "carries",
    "dribbles",
    "dribbles_completed",
    "fouls_won",
    "dispossessed",
    "miscontrols",
    "errors",
    "offsides",
    # Carries and dribbles p90
    "carries_p90",
    "dribbles_p90",
    "dribbles_completed_p90",
    "fouls_won_p90",
    "dispossessed_p90",
    "miscontrols_p90",
    "errors_p90",
    "offsides_p90",
    # Carries and dribbles rates
    "dribble_success",
    # Defending
    "tackles",
    "interceptions",
    "clearances",
    "blocks",
    "pressures",
    "ball_recoveries",
    "counterpresses",
    "dribbled_past",
    "fifty_fifties",
    "shields",
    "fouls_committed",
    # Defending p90
    "tackles_p90",
    "interceptions_p90",
    "clearances_p90",
    "blocks_p90",
    "pressures_p90",
    "ball_recoveries_p90",
    "counterpresses_p90",
    "dribbled_past_p90",
    "fifty_fifties_p90",
    "shields_p90",
    "fouls_committed_p90",
    # Aerials
    "aerials",
    "aerials_won",
    # Aerials p90
    "aerials_p90",
    "aerials_won_p90",
    # Aerials rates
    "aerial_win_rate",
    # Cards
    "yellow_cards",
    "red_cards",
    # Cards p90
    "yellow_cards_p90",
    "red_cards_p90",
    # Foot share of left- and right-footed passes plus shots (headed actions excluded)
    "left_ratio",
    "right_ratio",
]

LINEUP_FIELDS = {"appearances", "starts", "minutes"}
RATE_FIELDS = {
    "shot_accuracy",
    "conversion",
    "xg_per_shot",
    "goals_minus_xg",
    "pass_completion",
    "dribble_success",
    "aerial_win_rate",
    "left_ratio",
    "right_ratio",
} | {field for field in STAT_FIELDS if field.endswith("_share")}
DERIVED_FIELDS = {field for field in STAT_FIELDS if field.endswith("_p90")} | RATE_FIELDS
# Summed across matches.
COUNT_FIELDS = [field for field in STAT_FIELDS if field not in DERIVED_FIELDS]
# Volume counts that get a `{name}_p90` companion.
P90_FIELDS = [field.removesuffix("_p90") for field in STAT_FIELDS if field.endswith("_p90")]


# 2. 🔧 Helper functions

Minutes come from lineup `positions` windows. Clock strings are `MM:SS`. `to` is null while the player is still on; that end is filled with the last event time of the match. Overlapping windows are merged so a tactical shift is not double-counted. A gap (player off, then on again) stays as two intervals and is added.

`load_match` stacks both clubs' lineups and keeps only the event columns used for counts. `event_stats` flags each event row, then sums those flags per `player_id`. Columns a match never uses (no shots, no cards) are filled with `optional_column`.


In [2]:
def _clock_to_seconds(value):
    """StatsBomb lineup clocks are MM:SS (match clock). None means still on."""
    if value is None:
        return None
    mins, secs = str(value).split(":")
    return float(mins) * 60 + float(secs)


def as_boolean_series(frame, col):
    """Boolean flags arrive as bool or as 'True' strings depending on source."""
    if col not in frame.columns:
        return pd.Series(False, index=frame.index)
    series = frame[col]
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype("string").str.lower().isin(["true", "1", "yes"])


def optional_column(frame, col, fill=np.nan):
    # Event columns are missing when that match has none of that event type.
    if col not in frame.columns:
        return pd.Series(fill, index=frame.index)
    return frame[col]


def appearance_info_from_lineups(positions_list, match_end_time):
    """Minutes, whether they started, and the StatsBomb role names in this match."""

    playing_time = []
    roles = set()
    started = False

    for position in positions_list or []:
        start = _clock_to_seconds(position.get("from"))

        # Only Starting XI counts as a start.
        if position.get("start_reason") == "Starting XI":
            started = True

        # `to` is null if they were still on at the whistle. Fill with last event.
        end = _clock_to_seconds(position.get("to"))
        if end is None:
            end = match_end_time

        playing_time.append((start, end))
        pos_name = str(position.get("position") or "").strip()
        if pos_name:
            roles.add(pos_name)

    # Empty list (unused sub): no appearance this match.
    if not playing_time:
        return 0.0, False, roles

    # Merge overlapping [from, to] so a tactical shift is not double-counted
    # in total minutes. Adjacent non-overlap playing_time stay separate.
    playing_time.sort()
    merged = [playing_time[0]]
    for start, end in playing_time[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    minutes = sum(end - start for start, end in merged) / 60.0
    return minutes, started, roles


def flatten_lineups(lineups, match_id):
    """statsbombpy returns {team_name: frame}. Stack into one match table."""
    parts = []
    for team_name, team in lineups.items():
        part = team.copy()
        part["team_name"] = team_name
        part["match_id"] = match_id
        parts.append(part)

    return pd.concat(parts, ignore_index=True)


# Event columns used for the season counts. Everything else is dropped in load_match.
EVENT_COLS = [
    "player_id",
    "player",
    "type",
    "shot_outcome",
    "shot_statsbomb_xg",
    "pass_outcome",
    "pass_goal_assist",
    "pass_shot_assist",
    "dribble_outcome",
    "duel_type",
    "foul_committed_card",
    "bad_behaviour_card",
    "minute",
    "second",
    "pass_body_part",
    "shot_body_part",
    "pass_height",
    "pass_aerial_won",
    "clearance_aerial_won",
    "shot_aerial_won",
    "miscontrol_aerial_won",
    "counterpress",
    "shot_first_time",
    "shot_one_on_one",
    "pass_cross",
    "pass_cut_back",
    "pass_switch",
    "pass_through_ball",
]


def load_match(match_id):
    lineups = flatten_lineups(sb.lineups(match_id=match_id), match_id)
    events = sb.events(match_id=match_id)
    keep = [col for col in EVENT_COLS if col in events.columns]
    needed_events = events.loc[:, keep].copy()
    return lineups, needed_events


def event_stats(events):
    """Per-player counts for one match."""
    # Half start/end, Starting XI and some referee events have no player_id. Drop them.
    player_event = events.dropna(subset=["player_id"]).copy()
    player_event["player_id"] = player_event["player_id"].astype(int)

    # Outcome columns are missing when that match has none of that event type.
    event_type = optional_column(player_event, "type").astype("string")
    shot_outcome = optional_column(player_event, "shot_outcome").astype("string")
    pass_outcome = optional_column(player_event, "pass_outcome").astype("string")
    dribble_outcome = optional_column(player_event, "dribble_outcome").astype("string")
    duel_type = optional_column(player_event, "duel_type").astype("string")
    pass_body_part = optional_column(player_event, "pass_body_part").astype("string")
    shot_body_part = optional_column(player_event, "shot_body_part").astype("string")
    pass_height = optional_column(player_event, "pass_height").astype("string")
    xg = pd.to_numeric(optional_column(player_event, "shot_statsbomb_xg", 0), errors="coerce").fillna(0)
    # Cards sit on Foul Committed or Bad Behaviour. Concatenate so either source counts.
    cards = (
        optional_column(player_event, "foul_committed_card").astype("string").fillna("")
        + "|"
        + optional_column(player_event, "bad_behaviour_card").astype("string").fillna("")
    )

    # Boolean (or numeric) columns, True on matching rows, then summed per player.
    is_pass = event_type == "Pass"
    is_shot = event_type == "Shot"
    aerial_won = (
        as_boolean_series(player_event, "pass_aerial_won")
        | as_boolean_series(player_event, "clearance_aerial_won")
        | as_boolean_series(player_event, "shot_aerial_won")
        | as_boolean_series(player_event, "miscontrol_aerial_won")
    )
    aerial_lost = (event_type == "Duel") & (duel_type == "Aerial Lost")

    out = player_event.assign(
        goals=(event_type == "Shot") & (shot_outcome == "Goal"),
        xg=xg,
        assists=as_boolean_series(player_event, "pass_goal_assist"),
        shots=is_shot,
        shots_on_target=is_shot & shot_outcome.isin({"Goal", "Saved", "Saved to Post"}),
        key_passes=as_boolean_series(player_event, "pass_shot_assist") | as_boolean_series(player_event, "pass_goal_assist"),
        passes=is_pass,
        passes_completed=is_pass & pass_outcome.isna(),
        carries=event_type == "Carry",
        dribbles=event_type == "Dribble",
        dribbles_completed=(event_type == "Dribble") & (dribble_outcome == "Complete"),
        tackles=(event_type == "Duel") & (duel_type == "Tackle"),
        interceptions=event_type == "Interception",
        clearances=event_type == "Clearance",
        blocks=event_type == "Block",
        pressures=event_type == "Pressure",
        ball_recoveries=event_type == "Ball Recovery",
        fouls_committed=event_type == "Foul Committed",
        fouls_won=event_type == "Foul Won",
        dispossessed=event_type == "Dispossessed",
        miscontrols=event_type == "Miscontrol",
        yellow_cards=cards.str.contains("Yellow", na=False),

        # Second yellow is a yellow and a dismissal.
        red_cards=cards.str.contains("Red Card", na=False)
        | cards.str.contains("Second Yellow", na=False),

        dribbled_past=event_type == "Dribbled Past",
        fifty_fifties=event_type == "50/50",
        errors=event_type == "Error",
        shields=event_type == "Shield",
        offsides=event_type == "Offside",
        aerials=aerial_won | aerial_lost,
        aerials_won=aerial_won,
        passes_left_foot=is_pass & (pass_body_part == "Left Foot"),
        passes_right_foot=is_pass & (pass_body_part == "Right Foot"),
        passes_head=is_pass & (pass_body_part == "Head"),
        shots_left_foot=is_shot & (shot_body_part == "Left Foot"),
        shots_right_foot=is_shot & (shot_body_part == "Right Foot"),
        shots_head=is_shot & (shot_body_part == "Head"),
        passes_ground=is_pass & (pass_height == "Ground Pass"),
        passes_low=is_pass & (pass_height == "Low Pass"),
        passes_high=is_pass & (pass_height == "High Pass"),
        crosses=is_pass & as_boolean_series(player_event, "pass_cross"),
        cut_backs=is_pass & as_boolean_series(player_event, "pass_cut_back"),
        switches=is_pass & as_boolean_series(player_event, "pass_switch"),
        through_balls=is_pass & as_boolean_series(player_event, "pass_through_ball"),
        counterpresses=as_boolean_series(player_event, "counterpress"),
        shots_first_time=is_shot & as_boolean_series(player_event, "shot_first_time"),
        shots_one_on_one=is_shot & as_boolean_series(player_event, "shot_one_on_one"),
    )
    # Appearances, starts and minutes come from lineups. Derived fields are not in this frame.
    metrics = [field for field in COUNT_FIELDS if field not in LINEUP_FIELDS]
    return out.groupby("player_id", as_index=False)[metrics].sum()


# 3. 🔁 Season loop

One pass over every match. Lineups update the player register, appearances, starts, and minutes. Events add that match's counts. Both frames are dropped before the next fetch.

Name and country are last-write. Clubs and roles are unique labels, not a primary team or position. Unused substitutes never enter the player base.


In [3]:
matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
match_ids = [int(mid) for mid in matches["match_id"]]

# Accumulators. One dict entry per player_id across the whole season.
# Name and country: last appearance wins. Clubs and roles: unique labels only.
player_name = {}
country = {}
teams_seen = defaultdict(set)
positions_seen = defaultdict(set)
season = defaultdict(
    lambda: {field: 0.0 for field in COUNT_FIELDS}
)
failed = []
loaded = 0

for match_id in tqdm(match_ids, desc="Matches"):
    try:
        lineups, events = load_match(match_id)
        loaded += 1
    except Exception as exc:
        failed.append(f"match {match_id}: {exc}")
        continue

    # Close position windows whose `to` is null (still on at the whistle).
    match_end_time = float((events["minute"] * 60 + events["second"]).max())
    # player_ids who got on the pitch. Event counts later must be in this set.
    appeared = set()

    # Lineup half
    for player_lineup_row in lineups.itertuples(index=False):
        positions_list = getattr(player_lineup_row, "positions", [])

        minutes, started, roles = appearance_info_from_lineups(positions_list, match_end_time)

        # Skip unused substitutes. They never enter the player base.
        if minutes <= 0:
            continue

        player_id = int(player_lineup_row.player_id)
        appeared.add(player_id)

        player_name[player_id] = player_lineup_row.player_name
        country[player_id] = player_lineup_row.country
        teams_seen[player_id].add(player_lineup_row.team_name)
        positions_seen[player_id].update(roles)

        player_season_stats = season[player_id]
        player_season_stats["appearances"] += 1
        player_season_stats["starts"] += int(started)
        player_season_stats["minutes"] += minutes

    # Event half
    stats_frame = event_stats(events)
    for player_game_stats in stats_frame.itertuples(index=False):

        player_id = int(player_game_stats.player_id)

        # Ignore anyone who did not get on the pitch.
        if player_id not in appeared:
            continue

        player_season_stats = season[player_id]
        for stat in COUNT_FIELDS:
            if stat not in LINEUP_FIELDS:
                player_season_stats[stat] += float(getattr(player_game_stats, stat, 0) or 0)

    # Free this match before fetching the next one.
    del lineups, events, stats_frame

print(f"Season {SEASON_LABEL}")
print(f"Matches in list: {len(match_ids)}")
print(f"Loaded {loaded} / failed {len(failed)}")
if failed:
    for item in failed:
        print(f"  {item}")


Matches:   0%|          | 0/380 [00:00<?, ?it/s]

Season 2015/2016
Matches in list: 380
Loaded 380 / failed 0


# 4. 📤 Assemble and write

One row per player who appeared. `teams` and `positions` are unique labels, sorted alphabetically. Count columns are rounded to integers; minutes keep one decimal and xG three.

Volume counts (except appearances, starts, minutes, and body-part / pass-height mix) are also stored per 90 minutes as `{stat}_p90`. Success rates and mix shares (`pass_completion`, `dribble_success`, `aerial_win_rate`, `shot_accuracy`, `conversion`, `xg_per_shot`, `goals_minus_xg`, body-part, pass-height, and cross / cut-back / switch / through-ball shares) are independent of minutes. A zero denominator (no shots, no passes, no dribbles, no aerials) is stored as 0, not blank. `left_ratio` and `right_ratio` are the share of left- and right-footed passes plus shots; headed actions are left out of the denominator. `goals / xg` is not stored: it is unstable on small shot samples. Use `goals_minus_xg` instead.

Output: `projects/data/statsbomb/pl_1516_player_season_stats.csv`.


In [4]:
rows = []
for player_id, stats in season.items():
    if stats["appearances"] <= 0:
        continue
    teams = sorted(teams_seen[player_id])
    roles = sorted(positions_seen[player_id])
    rows.append(
        {
            "player_id": player_id,
            "player_name": player_name.get(player_id, ""),
            "country": country.get(player_id, ""),
            "teams": ", ".join(teams),
            "positions": ", ".join(roles),
            **stats,
        }
    )

df = pd.DataFrame(rows)
df["player_id"] = df["player_id"].astype(int)
df["minutes"] = df["minutes"].round(1)
df["xg"] = df["xg"].round(3)
for col in COUNT_FIELDS:
    if col in {"minutes", "xg"}:
        continue
    df[col] = df[col].round().astype(int)
df = df.sort_values(["minutes", "player_name"], ascending=[False, True]).reset_index(
    drop=True
)

minutes = df["minutes"].replace(0, np.nan)
for col in P90_FIELDS:
    df[f"{col}_p90"] = (df[col] * 90.0 / minutes).round(3)


def _share(numer, denom):
    return (numer / denom.replace(0, np.nan)).fillna(0).round(3)


df["shot_accuracy"] = _share(df["shots_on_target"], df["shots"])
df["conversion"] = _share(df["goals"], df["shots"])
df["xg_per_shot"] = _share(df["xg"], df["shots"])
df["goals_minus_xg"] = (df["goals"] - df["xg"]).round(3)
df["shot_left_share"] = _share(df["shots_left_foot"], df["shots"])
df["shot_right_share"] = _share(df["shots_right_foot"], df["shots"])
df["shot_head_share"] = _share(df["shots_head"], df["shots"])

df["pass_completion"] = _share(df["passes_completed"], df["passes"])
df["pass_left_share"] = _share(df["passes_left_foot"], df["passes"])
df["pass_right_share"] = _share(df["passes_right_foot"], df["passes"])
df["pass_head_share"] = _share(df["passes_head"], df["passes"])
df["pass_ground_share"] = _share(df["passes_ground"], df["passes"])
df["pass_low_share"] = _share(df["passes_low"], df["passes"])
df["pass_high_share"] = _share(df["passes_high"], df["passes"])
df["cross_share"] = _share(df["crosses"], df["passes"])
df["cut_back_share"] = _share(df["cut_backs"], df["passes"])
df["switch_share"] = _share(df["switches"], df["passes"])
df["through_ball_share"] = _share(df["through_balls"], df["passes"])

df["dribble_success"] = _share(df["dribbles_completed"], df["dribbles"])
df["aerial_win_rate"] = _share(df["aerials_won"], df["aerials"])

footed = (
    df["passes_left_foot"]
    + df["passes_right_foot"]
    + df["shots_left_foot"]
    + df["shots_right_foot"]
)
df["left_ratio"] = _share(df["passes_left_foot"] + df["shots_left_foot"], footed)
df["right_ratio"] = _share(df["passes_right_foot"] + df["shots_right_foot"], footed)

id_cols = ["player_id", "player_name", "country", "teams", "positions"]
df = df[id_cols + STAT_FIELDS]

# Write player season stats to CSV
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_PATH, index=False)
print(f"Players (appeared): {len(df)}")
print(f"Wrote {DATA_PATH}")
display(df.head())


Players (appeared): 550
Wrote data\statsbomb\pl_1516_player_season_stats.csv


,player_id,player_name,country,teams,positions,appearances,starts,minutes,goals,xg,...,aerials_won,aerials_p90,aerials_won_p90,aerial_win_rate,yellow_cards,red_cards,yellow_cards_p90,red_cards_p90,left_ratio,right_ratio
0,3815,Kasper Schmeichel,Denmark,Leicester City,Goalkeeper,38,38,3594.0,0,0.000,...,0,0.000,0.000,0.000,2,0,0.050,0.000,0.031,0.969
1,3813,Wes Morgan,Jamaica,Leicester City,"Left Center Back, Right Center Back",38,38,3594.0,2,2.948,...,129,5.008,3.230,0.645,3,0,0.075,0.000,0.135,0.865
2,3608,Simon Francis,England,AFC Bournemouth,"Left Center Midfield, Right Back, Right Center...",38,38,3591.6,0,0.197,...,137,5.463,3.433,0.628,5,1,0.125,0.025,0.142,0.858
3,3344,Andrew Surman,England,AFC Bournemouth,"Center Defensive Midfield, Left Center Midfiel...",38,38,3590.9,0,0.762,...,78,3.734,1.955,0.523,4,0,0.100,0.000,0.839,0.161
4,20005,Toby Alderweireld,Belgium,Tottenham Hotspur,"Right Back, Right Center Back",38,38,3569.2,4,3.184,...,89,3.883,2.244,0.578,3,0,0.076,0.000,0.164,0.836
